In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Configuration

In [ ]:

TRAIN_DIR = '/content/drive/MyDrive/Colab/Skin Cancer classification neural networks/Skin cancer ISIC The International Skin Imaging Collaboration/Train'
TEST_DIR = '/content/drive/MyDrive/Colab/Skin Cancer classification neural networks/Skin cancer ISIC The International Skin Imaging Collaboration/Test'
IMG_WIDTH = 100
IMG_HEIGHT = 75
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
NUM_WORKERS = 4
MODEL_PATH = './best_model.pth'



#  Dataset Class

In [ ]:

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_path = self.df.iloc[index]['image_path']
        label = self.df.iloc[index]['label']
        image = Image.open(img_path).convert('RGB').resize((IMG_WIDTH, IMG_HEIGHT))

        if self.transform:
            image = self.transform(image)

        return image, label



#  Load and Balance Data


In [ ]:
def create_dataframe(image_dir):
    data = []
    for idx, class_name in enumerate(os.listdir(image_dir)):
        class_path = os.path.join(image_dir, class_name)
        for fname in os.listdir(class_path):
            data.append({'image_path': os.path.join(class_path, fname), 'label': idx})
    return pd.DataFrame(data)

train_df = create_dataframe(TRAIN_DIR)
test_df = create_dataframe(TEST_DIR)
full_df = pd.concat([train_df, test_df], ignore_index=True)
max_per_class = 2000
balanced_df = full_df.groupby('label').head(max_per_class).sample(frac=1, random_state=42).reset_index(drop=True)
num_classes = len(balanced_df['label'].unique())
label_map = {i: name for i, name in enumerate(sorted(os.listdir(TRAIN_DIR)))}

#  Transforms and Loaders


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_data, val_data = train_test_split(balanced_df, test_size=0.2, stratify=balanced_df['label'], random_state=42)
train_dataset = SkinDataset(train_data, transform=train_transform)
val_dataset = SkinDataset(val_data, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


# LOAD PRETRAINED MODEL

In [ ]:
model = models.densenet201(pretrained=True)
for param in model.features.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(model.classifier.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes)
)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth
100%|██████████| 77.4M/77.4M [00:00<00:00, 101MB/s]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [ ]:
# ======================== Loss and Optimizer ========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

# TRAINING LOOP

In [ ]:
best_val_acc = 0
for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / len(train_loader.dataset)

    # Validation
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = val_correct / len(val_loader.dataset)
    print(f"Epoch {epoch+1} - Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        torch.save(model.state_dict(), MODEL_PATH)
        best_val_acc = val_acc
        print("Model saved!")



Validating: 100%|██████████| 15/15 [00:29<00:00,  1.95s/it]


Epoch 1 - Train Acc: 0.5705 | Val Acc: 0.4937
Model saved!


Validating: 100%|██████████| 15/15 [00:29<00:00,  1.97s/it]


Epoch 2 - Train Acc: 0.5790 | Val Acc: 0.4810


Validating: 100%|██████████| 15/15 [00:29<00:00,  1.99s/it]


...

Validating: 100%|██████████| 15/15 [00:31<00:00,  2.07s/it]


Epoch 75 - Train Acc: 0.6561 | Val Acc: 0.5063


Validating: 100%|██████████| 15/15 [00:29<00:00,  1.95s/it]


Epoch 76 - Train Acc: 0.6550 | Val Acc: 0.4873


Epoch 77/100 - Training:  43%|████▎     | 26/60 [01:05<01:17,  2.27s/it]

#  Prediction


In [ ]:
def predict_image(img_path):
    model.load_state_dict(torch.load(MODEL_PATH))
    model.eval()
    image_obj = Image.open(img_path).convert('RGB').resize((IMG_WIDTH, IMG_HEIGHT))
    img_tensor = test_transform(image_obj).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        predicted = output.argmax(1).item()

    return predicted

#  Run Prediction


In [ ]:
sample_image_path = '/content/drive/MyDrive/Colab/Skin Cancer classification neural networks/Skin cancer ISIC The International Skin Imaging Collaboration/Test/nevus/ISIC_0000005.jpg'
predicted_class_index = predict_image(sample_image_path)
predicted_class_name = label_map[predicted_class_index]

print(f"\n🔍 Predicted Class: {predicted_class_name}")


🔍 Predicted Class: melanoma
